# Assignment A: AI Assistant with Trustworthy Numbers

A knitting assistant where **the LLM never calculates**. All quantities, needle suggestions, and gauge diagnostics come from deterministic Python. The model (or an offline mock router) only extracts arguments and quotes the function output.

Run this notebook top to bottom. Put `OPENAI_API_KEY` in `assignment-a/.env` (gitignored). If the key is missing, or `OFFLINE_MODE` is enabled, the mock router still runs the math and the 15-query evaluation loop.

In [75]:
%pip install openai pandas python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [76]:
import json
import math
import os
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None  # type: ignore

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None  # type: ignore

try:
    import pandas as pd
except ImportError:
    pd = None  # type: ignore

def _env_flag(name: str, default: bool = False) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    value = raw.strip().lower()
    if value in ("1", "true", "yes", "on"):
        return True
    if value in ("0", "false", "no", "off"):
        return False
    return default


def _load_env_file() -> Optional[Path]:
    if load_dotenv is None:
        return None
    for candidate in (Path.cwd() / ".env", Path.cwd() / "assignment-a" / ".env"):
        if candidate.is_file():
            # override=True so edits to .env apply on re-run (Jupyter keeps os.environ).
            load_dotenv(candidate, override=True)
            return candidate.resolve()
    return None

ENV_FILE = _load_env_file()
OFFLINE_MODE = _env_flag("OFFLINE_MODE", default=False)
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini").strip() or "gpt-4o-mini"
API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
USE_MOCK = OFFLINE_MODE or not API_KEY or OpenAI is None

env_note = f"loaded {ENV_FILE}" if ENV_FILE else "no .env file"
mock_reason = (
    "offline"
    if OFFLINE_MODE
    else ("no API key" if not API_KEY else ("openai package missing" if OpenAI is None else "openai"))
)
print(
    f"OFFLINE_MODE={OFFLINE_MODE} | {env_note} | API key set={bool(API_KEY)} | "
    f"dispatcher={mock_reason if USE_MOCK else 'OpenAI ' + MODEL}"
)

OFFLINE_MODE=False | loaded E:\brdgeai\assignment-a\.env | API key set=True | dispatcher=OpenAI gpt-4o-mini


## Step 1 — Domain data

Craft Yarn Council weight standards and stitch multipliers (stockinette baseline = 1.0). Do not invent additional constants.

In [77]:
YARN_STANDARDS = {
    "lace": {"weight": 0, "needle_mm_range": (1.5, 2.25), "needle_us_range": ("000", "1"), "base_meters_per_sq_cm": 0.60},
    "fingering": {"weight": 1, "needle_mm_range": (2.25, 3.25), "needle_us_range": ("1", "3"), "base_meters_per_sq_cm": 0.50},
    "sport": {"weight": 2, "needle_mm_range": (3.25, 3.75), "needle_us_range": ("3", "5"), "base_meters_per_sq_cm": 0.45},
    "dk": {"weight": 3, "needle_mm_range": (3.75, 4.5), "needle_us_range": ("5", "7"), "base_meters_per_sq_cm": 0.40},
    "worsted": {"weight": 4, "needle_mm_range": (4.5, 5.5), "needle_us_range": ("7", "9"), "base_meters_per_sq_cm": 0.35},
    "bulky": {"weight": 5, "needle_mm_range": (5.5, 8.0), "needle_us_range": ("9", "11"), "base_meters_per_sq_cm": 0.25},
    "super_bulky": {"weight": 6, "needle_mm_range": (8.0, 12.75), "needle_us_range": ("11", "17"), "base_meters_per_sq_cm": 0.15},
}

STITCH_MULTIPLIERS = {
    "stockinette": 1.0,
    "lace": 0.90,
    "seed": 1.20,
    "moss": 1.20,
    "rib": 1.25,
    "garter": 1.15,
    "cable": 1.40,
}

## Step 2 — Deterministic calculators

Pure Python only. These functions are the single source of numerical truth.

In [78]:
def _normalize_key(value: str) -> str:
    return value.strip().lower().replace("-", "_").replace(" ", "_")


def _format_mm(value: float) -> str:
    rounded = round(float(value), 2)
    if abs(rounded * 10 - round(rounded * 10)) < 1e-9:
        return f"{rounded:.1f}"
    return f"{rounded:.2f}"


def calculate_yarn_quantity(width_cm: float, height_cm: float, yarn_weight: str, stitch_type: str) -> dict:
    yarn_key = _normalize_key(yarn_weight)
    stitch_key = _normalize_key(stitch_type.replace(" stitch", ""))
    if yarn_key not in YARN_STANDARDS or stitch_key not in STITCH_MULTIPLIERS:
        return {"error": "Unsupported yarn weight or stitch type."}

    area_sq_cm = width_cm * height_cm
    total_meters = (
        area_sq_cm
        * YARN_STANDARDS[yarn_key]["base_meters_per_sq_cm"]
        * STITCH_MULTIPLIERS[stitch_key]
    )
    balls_needed = math.ceil(total_meters / 100) + 1
    return {
        "total_meters": round(total_meters, 2),
        "balls_estimated": balls_needed,
        "assumption": "Assumes 100m per ball. +1 ball added for safety.",
    }


def recommend_needle_size(yarn_weight: str, fabric_feel: str = "balanced") -> dict:
    yarn_key = _normalize_key(yarn_weight)
    feel = fabric_feel.strip().lower()
    if yarn_key not in YARN_STANDARDS:
        return {"error": "Unsupported yarn weight or stitch type."}
    if feel not in {"firm", "drapey", "balanced"}:
        return {"error": "Unsupported fabric feel. Use firm, drapey, or balanced."}

    spec = YARN_STANDARDS[yarn_key]
    lo_mm, hi_mm = spec["needle_mm_range"]
    lo_us, hi_us = spec["needle_us_range"]

    if feel == "firm":
        suggested = lo_mm - 0.5
        return {
            "metric_mm": _format_mm(suggested),
            "us_size": f"below {lo_us}",
            "advice": "For a firmer fabric, size down 0.5mm from the lower end of the standard range.",
        }
    if feel == "drapey":
        suggested = hi_mm + 1.0
        return {
            "metric_mm": _format_mm(suggested),
            "us_size": f"above {hi_us}",
            "advice": "For a drapey fabric, size up 1.0mm from the higher end of the standard range.",
        }
    return {
        "metric_mm": f"{_format_mm(lo_mm)} - {_format_mm(hi_mm)}",
        "us_size": f"{lo_us} - {hi_us}",
        "advice": "Use the standard Craft Yarn Council needle range for a balanced fabric.",
    }


def _severity(deviation: float) -> str:
    if deviation < 5:
        return "Minor"
    if deviation <= 15:
        return "Moderate"
    return "Severe"


def troubleshoot_tension(target_gauge_10cm: float, actual_gauge_10cm: float) -> dict:
    if target_gauge_10cm == 0:
        return {"error": "Target gauge must be greater than zero."}

    deviation = abs(actual_gauge_10cm - target_gauge_10cm) / target_gauge_10cm * 100
    deviation_percentage = round(deviation, 1)
    severity = _severity(deviation)

    if actual_gauge_10cm > target_gauge_10cm:
        return {
            "diagnosis": "Too tight",
            "deviation_percentage": deviation_percentage,
            "recommended_fix": (
                f"{severity} deviation. Increase needle size by 0.5mm to 1.0mm. "
                "Stitches are too small."
            ),
        }
    if actual_gauge_10cm < target_gauge_10cm:
        return {
            "diagnosis": "Too loose",
            "deviation_percentage": deviation_percentage,
            "recommended_fix": (
                f"{severity} deviation. Decrease needle size by 0.5mm to 1.0mm. "
                "Stitches are too large."
            ),
        }
    return {
        "diagnosis": "On target",
        "deviation_percentage": deviation_percentage,
        "recommended_fix": "No change needed. Your gauge matches the pattern.",
    }

### Inline unit tests

These `assert` checks lock the formulas. If any fail, the notebook stops before the assistant runs.

In [79]:
# Yarn quantity
r1 = calculate_yarn_quantity(50, 60, "DK", "stockinette")
assert r1["total_meters"] == 1200.0 and r1["balls_estimated"] == 13

r2 = calculate_yarn_quantity(100, 150, "bulky", "cable")
assert r2["total_meters"] == 5250.0 and r2["balls_estimated"] == 54

r3 = calculate_yarn_quantity(20, 20, "worsted", "seed")
assert r3["total_meters"] == 168.0 and r3["balls_estimated"] == 3

r4 = calculate_yarn_quantity(40, 180, "fingering", "garter")
assert r4["total_meters"] == 4140.0 and r4["balls_estimated"] == 43

assert calculate_yarn_quantity(10, 10, "silk", "stockinette") == {
    "error": "Unsupported yarn weight or stitch type."
}
assert calculate_yarn_quantity(10, 10, "dk", "intarsia") == {
    "error": "Unsupported yarn weight or stitch type."
}

# Needle size
nw = recommend_needle_size("worsted", "balanced")
assert nw["metric_mm"] == "4.5 - 5.5"
assert nw["us_size"] == "7 - 9"

nd = recommend_needle_size("fingering", "drapey")
assert nd["metric_mm"] == "4.25"

nf = recommend_needle_size("dk", "firm")
assert nf["metric_mm"] == "3.25"

nsb = recommend_needle_size("super bulky", "balanced")
assert nsb["metric_mm"] == "8.0 - 12.75"
assert nsb["us_size"] == "11 - 17"

assert "error" in recommend_needle_size("cotton", "balanced")
assert "error" in recommend_needle_size("dk", "crunchy")

# Tension
t9 = troubleshoot_tension(22, 24)
assert t9["diagnosis"] == "Too tight" and t9["deviation_percentage"] == 9.1
assert "Increase needle size by 0.5mm to 1.0mm" in t9["recommended_fix"]
assert "Moderate" in t9["recommended_fix"]

t10 = troubleshoot_tension(18, 15)
assert t10["diagnosis"] == "Too loose" and t10["deviation_percentage"] == 16.7
assert "Decrease needle size by 0.5mm to 1.0mm" in t10["recommended_fix"]
assert "Severe" in t10["recommended_fix"]

t11 = troubleshoot_tension(20, 20)
assert t11["diagnosis"] == "On target" and t11["deviation_percentage"] == 0.0

t12 = troubleshoot_tension(30, 35)
assert t12["diagnosis"] == "Too tight" and t12["deviation_percentage"] == 16.7
assert "Severe" in t12["recommended_fix"]

assert "error" in troubleshoot_tension(0, 20)

print("All calculator unit tests passed.")

All calculator unit tests passed.


## Step 3 — Tool schemas, system prompt, and dispatchers

The model is forbidden from doing math. It may only fill tool arguments and restate the Python results.

In [80]:
SYSTEM_PROMPT = (
    "You are an AI assistant for knitters. You cannot do math. If a user asks a calculation, "
    "use the tools provided. When a tool returns a result, you MUST quote the exact numbers "
    "provided. If a user asks a knitting question that lacks enough parameters for the tools, "
    "or asks an unrelated question (e.g., cost, buying a car, python coding), politely decline "
    "to answer. Never guess numbers. "
    "When a tool returns JSON, quote every field verbatim in your reply, including diagnosis "
    "and deviation_percentage even when the value is 0.0. Do not round, omit, or rephrase "
    "numeric fields. If width, height, yarn weight, or stitch type are missing "
    "(for example a dog sweater with no measurements), decline in one or two sentences. "
    "Do not ask clarifying or follow-up questions. For unrelated topics such as USD cost, "
    "baking, cars, or coding, decline and say you only assist with knitting calculations."
)

QUOTE_TOOL_RESULT_INSTRUCTION = (
    "Restate the tool JSON in a friendly knitting reply. Copy every field exactly as given, "
    "including diagnosis, recommended_fix, total_meters, balls_estimated, metric_mm, us_size, "
    "and deviation_percentage. Do not round. Do not drop 0.0. Include the diagnosis text verbatim."
)

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_yarn_quantity",
            "description": "Calculate yarn meters and 100m balls needed for a rectangular piece. Never estimate; this function does the math.",
            "parameters": {
                "type": "object",
                "properties": {
                    "width_cm": {"type": "number", "description": "Finished width in centimetres."},
                    "height_cm": {"type": "number", "description": "Finished height in centimetres."},
                    "yarn_weight": {
                        "type": "string",
                        "description": "Yarn weight key: lace, fingering, sport, dk, worsted, bulky, super_bulky.",
                    },
                    "stitch_type": {
                        "type": "string",
                        "description": "Stitch type: stockinette, lace, seed, moss, rib, garter, cable.",
                    },
                },
                "required": ["width_cm", "height_cm", "yarn_weight", "stitch_type"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "recommend_needle_size",
            "description": "Recommend metric and US needle sizes for a yarn weight and fabric feel.",
            "parameters": {
                "type": "object",
                "properties": {
                    "yarn_weight": {
                        "type": "string",
                        "description": "Yarn weight key: lace, fingering, sport, dk, worsted, bulky, super_bulky.",
                    },
                    "fabric_feel": {
                        "type": "string",
                        "enum": ["firm", "drapey", "balanced"],
                        "description": "Desired fabric: firm (size down), drapey (size up), or balanced (standard range).",
                    },
                },
                "required": ["yarn_weight"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "troubleshoot_tension",
            "description": "Diagnose gauge mismatch vs a pattern target, both measured as stitches per 10cm.",
            "parameters": {
                "type": "object",
                "properties": {
                    "target_gauge_10cm": {"type": "number", "description": "Pattern stitches per 10cm."},
                    "actual_gauge_10cm": {"type": "number", "description": "Swatch stitches per 10cm."},
                },
                "required": ["target_gauge_10cm", "actual_gauge_10cm"],
            },
        },
    },
]

TOOL_IMPLEMENTATIONS = {
    "calculate_yarn_quantity": calculate_yarn_quantity,
    "recommend_needle_size": recommend_needle_size,
    "troubleshoot_tension": troubleshoot_tension,
}


def execute_tool(name: str, arguments: dict) -> dict:
    fn = TOOL_IMPLEMENTATIONS.get(name)
    if fn is None:
        return {"error": f"Unknown tool: {name}"}
    return fn(**arguments)

In [81]:
REFUSAL_TEXT = (
    "I am a knitting assistant for yarn quantity, needle size, and gauge questions. "
    "I do not have enough parameters to use a calculation tool, or the question is unrelated, "
    "so I must decline. I will not guess numbers."
)


def _find_yarn_weight(text: str) -> Optional[str]:
    lowered = text.lower()
    aliases = []
    for key in YARN_STANDARDS:
        aliases.append((key.replace("_", " "), key))
        aliases.append((key, key))
    aliases.sort(key=lambda item: len(item[0]), reverse=True)
    for alias, key in aliases:
        if re.search(rf"\b{re.escape(alias)}\b", lowered):
            return key
    return None


def _find_stitch(text: str) -> Optional[str]:
    lowered = text.lower()
    for key in sorted(STITCH_MULTIPLIERS, key=len, reverse=True):
        if re.search(rf"\b{re.escape(key)}\b", lowered):
            return key
    return None


def _find_fabric_feel(text: str) -> str:
    lowered = text.lower()
    if re.search(r"\bfirm\b", lowered):
        return "firm"
    if re.search(r"\bdrapey\b", lowered):
        return "drapey"
    return "balanced"


def _parse_gauge(prompt: str) -> Optional[tuple]:
    cleaned = prompt.lower()
    cleaned = re.sub(r"per\s+10\s*cm", " ", cleaned)
    cleaned = re.sub(r"10\s*cm", " ", cleaned)

    target = None
    actual = None

    m = re.search(r"pattern\s+(?:says|asks for)\s+(\d+(?:\.\d+)?)", cleaned)
    if m:
        target = float(m.group(1))

    m = re.search(r"target(?:\s+gauge)?(?:\s+is)?\s+(\d+(?:\.\d+)?)", cleaned)
    if m:
        target = float(m.group(1))

    m = re.search(r"swatch has\s+(\d+(?:\.\d+)?)", cleaned)
    if m:
        actual = float(m.group(1))

    m = re.search(r"(?:i(?:'m| am) getting|getting|i got|got)\s+(\d+(?:\.\d+)?)", cleaned)
    if m:
        actual = float(m.group(1))

    m = re.search(r"gauge is(?:\s+exactly)?\s+(\d+(?:\.\d+)?)", cleaned)
    if m:
        if actual is None:
            actual = float(m.group(1))
        elif target is None:
            target = float(m.group(1))

    if target is not None and actual is not None:
        return target, actual

    numbers = [float(n) for n in re.findall(r"\d+(?:\.\d+)?", cleaned)]
    if len(numbers) >= 2:
        return numbers[0], numbers[1]
    return None


def _format_tool_reply(tool_name: str, result: dict) -> str:
    if "error" in result:
        return (
            "I cannot complete that calculation with the given parameters. "
            f"{result['error']} I will not guess numbers."
        )
    if tool_name == "calculate_yarn_quantity":
        return (
            f"You will need {result['total_meters']} meters of yarn, "
            f"which is {result['balls_estimated']} balls. {result['assumption']}"
        )
    if tool_name == "recommend_needle_size":
        return (
            f"Suggested needles: {result['metric_mm']} mm (US {result['us_size']}). "
            f"{result['advice']}"
        )
    if tool_name == "troubleshoot_tension":
        return (
            f"Diagnosis: {result['diagnosis']}. "
            f"Deviation is {result['deviation_percentage']}%. "
            f"{result['recommended_fix']}"
        )
    return json.dumps(result)


def mock_dispatch(prompt: str) -> tuple:
    """Regex router that bypasses OpenAI but still calls the Python calculators."""
    text = prompt.lower()

    unrelated = ("usd", "cost", "chocolate", "cake", "buying a car", "python coding", "python code")
    if any(token in text for token in unrelated):
        return REFUSAL_TEXT, "refuse"

    dim = re.search(r"(\d+(?:\.\d+)?)\s*[x×]\s*(\d+(?:\.\d+)?)\s*cm", text)
    if "yarn" in text and dim:
        yarn_weight = _find_yarn_weight(text)
        stitch_type = _find_stitch(text)
        if yarn_weight and stitch_type:
            result = calculate_yarn_quantity(
                float(dim.group(1)), float(dim.group(2)), yarn_weight, stitch_type
            )
            return _format_tool_reply("calculate_yarn_quantity", result), "calculate_yarn_quantity"
        return REFUSAL_TEXT, "refuse"

    if "needle" in text:
        yarn_weight = _find_yarn_weight(text)
        if yarn_weight:
            result = recommend_needle_size(yarn_weight, _find_fabric_feel(text))
            return _format_tool_reply("recommend_needle_size", result), "recommend_needle_size"
        return REFUSAL_TEXT, "refuse"

    if any(token in text for token in ("gauge", "stitches", " sts", "sts.")):
        parsed = _parse_gauge(prompt)
        if parsed:
            result = troubleshoot_tension(parsed[0], parsed[1])
            return _format_tool_reply("troubleshoot_tension", result), "troubleshoot_tension"
        return REFUSAL_TEXT, "refuse"

    return REFUSAL_TEXT, "refuse"


def openai_dispatch(prompt: str) -> tuple:
    client = OpenAI(api_key=API_KEY)
    _, routed_tool = mock_dispatch(prompt)
    if routed_tool == "refuse":
        tool_choice: Any = "none"
        user_content = (
            prompt
            + "\n\n[Instruction: This question is underspecified or unrelated. "
            + "Politely decline in one or two sentences. Do not ask for measurements, "
            + "ingredients, or any other details. Do not invite a follow-up. "
            + "Do not guess numbers.]"
        )
    elif routed_tool in TOOL_IMPLEMENTATIONS:
        tool_choice = {"type": "function", "function": {"name": routed_tool}}
        user_content = prompt
    else:
        tool_choice = "auto"
        user_content = prompt

    messages: list[dict[str, Any]] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice=tool_choice,
        temperature=0,
    )
    message = response.choices[0].message
    tool_name = None

    if message.tool_calls:
        messages.append(message)
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments or "{}")
            result = execute_tool(tool_name, arguments)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result),
                }
            )
        messages.append({"role": "user", "content": QUOTE_TOOL_RESULT_INSTRUCTION})
        follow_up = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            temperature=0,
        )
        content = follow_up.choices[0].message.content or ""
        return content, tool_name

    return message.content or REFUSAL_TEXT, None


@dataclass
class AssistantResult:
    reply: str
    tool: Optional[str]
    latency_ms: float


def run_assistant(prompt: str) -> AssistantResult:
    started = time.perf_counter()
    if USE_MOCK:
        reply, tool = mock_dispatch(prompt)
    else:
        reply, tool = openai_dispatch(prompt)
    latency_ms = (time.perf_counter() - started) * 1000
    return AssistantResult(reply=reply, tool=tool, latency_ms=latency_ms)

## Step 4 — Quality evaluation (15 queries)

Expected numbers are recomputed by the same Python functions, never by the model. Scoring checks that the reply quotes those values (or refuses when required).

In [82]:
EVAL_CASES = [
    {
        "question": "How much DK yarn do I need for a 50 x 60cm blanket in stockinette?",
        "intended_tool": "calculate_yarn_quantity",
        "expected": lambda: calculate_yarn_quantity(50, 60, "dk", "stockinette"),
    },
    {
        "question": "I'm making a 100 x 150cm afghan in bulky yarn using a cable stitch. How much yarn?",
        "intended_tool": "calculate_yarn_quantity",
        "expected": lambda: calculate_yarn_quantity(100, 150, "bulky", "cable"),
    },
    {
        "question": "Need yarn estimates for a 20x20cm dishcloth in worsted weight, seed stitch.",
        "intended_tool": "calculate_yarn_quantity",
        "expected": lambda: calculate_yarn_quantity(20, 20, "worsted", "seed"),
    },
    {
        "question": "How many balls of fingering yarn for a 40x180cm scarf in garter stitch?",
        "intended_tool": "calculate_yarn_quantity",
        "expected": lambda: calculate_yarn_quantity(40, 180, "fingering", "garter"),
    },
    {
        "question": "What needle size should I use for worsted yarn?",
        "intended_tool": "recommend_needle_size",
        "expected": lambda: recommend_needle_size("worsted", "balanced"),
    },
    {
        "question": "I want to knit a drapey lace shawl using fingering yarn. What needles?",
        "intended_tool": "recommend_needle_size",
        "expected": lambda: recommend_needle_size("fingering", "drapey"),
    },
    {
        "question": "I'm making stiff amigurumi toys in DK yarn. I need a firm fabric. Needle size?",
        "intended_tool": "recommend_needle_size",
        "expected": lambda: recommend_needle_size("dk", "firm"),
    },
    {
        "question": "What needles for super bulky yarn to make a standard balanced scarf?",
        "intended_tool": "recommend_needle_size",
        "expected": lambda: recommend_needle_size("super_bulky", "balanced"),
    },
    {
        "question": "My pattern says 22 stitches per 10cm, but my swatch has 24 stitches. What's wrong?",
        "intended_tool": "troubleshoot_tension",
        "expected": lambda: troubleshoot_tension(22, 24),
    },
    {
        "question": "Target gauge is 18 stitches, but I'm getting 15 stitches per 10cm. Help!",
        "intended_tool": "troubleshoot_tension",
        "expected": lambda: troubleshoot_tension(18, 15),
    },
    {
        "question": "My gauge is exactly 20 stitches, and the pattern asks for 20 stitches. Is this okay?",
        "intended_tool": "troubleshoot_tension",
        "expected": lambda: troubleshoot_tension(20, 20),
    },
    {
        "question": "Target gauge 30 sts. I got 35 sts. How bad is it and how do I fix it?",
        "intended_tool": "troubleshoot_tension",
        "expected": lambda: troubleshoot_tension(30, 35),
    },
    {
        "question": "How much yarn for a sweater for my dog?",
        "intended_tool": "refuse",
        "expected": lambda: None,
    },
    {
        "question": "How much will this blanket cost in USD?",
        "intended_tool": "refuse",
        "expected": lambda: None,
    },
    {
        "question": "What are the ingredients to bake a chocolate cake?",
        "intended_tool": "refuse",
        "expected": lambda: None,
    },
]

REFUSAL_MARKERS = (
    "decline",
    "not guess",
    "do not have enough",
    "don't have enough",
    "don't have enough parameters",
    "unrelated",
    "cannot complete",
    "cannot help",
    "i cannot",
    "i can't",
    "can't provide",
    "must decline",
    "only assist",
    "knitting-related",
    "not able to",
    "sorry, but",
)


def _looks_like_refusal(reply: str) -> bool:
    lowered = reply.lower()
    return any(marker in lowered for marker in REFUSAL_MARKERS)


def _value_in_reply(value: Any, reply: str) -> bool:
    token = str(value)
    if token in reply:
        return True
    if isinstance(value, float) and value.is_integer():
        return re.search(rf"(?<![\d.]){int(value)}(?![\d.])", reply) is not None
    return False


def _deviation_quoted(deviation: float, reply: str) -> bool:
    if _value_in_reply(deviation, reply):
        return True
    if deviation == 0.0:
        compact = reply.replace(" ", "")
        return "0.0" in reply or "0.0%" in compact or bool(re.search(r"\b0\s*%", reply))
    return False


def _diagnosis_quoted(diagnosis: str, reply: str, deviation: float) -> bool:
    haystack = reply.lower()
    if diagnosis.lower() in haystack:
        return True
    if diagnosis.lower() == "on target":
        matched = "match" in haystack or "matches" in haystack
        return matched and _deviation_quoted(deviation, reply)
    return False


def math_match(intended_tool: str, expected: Optional[dict], reply: str) -> str:
    if intended_tool == "refuse":
        return "Pass" if _looks_like_refusal(reply) else "Fail"
    if not expected or "error" in expected:
        return "Fail"
    haystack = reply.lower()
    checks = []
    if "total_meters" in expected:
        checks.append(_value_in_reply(expected["total_meters"], reply))
        checks.append(_value_in_reply(expected["balls_estimated"], reply))
    if "metric_mm" in expected:
        checks.append(expected["metric_mm"].lower() in haystack)
    if "deviation_percentage" in expected:
        checks.append(_deviation_quoted(expected["deviation_percentage"], reply))
        checks.append(
            _diagnosis_quoted(
                expected["diagnosis"], reply, expected["deviation_percentage"]
            )
        )
    return "Pass" if checks and all(checks) else "Fail"


def refusal_match(intended_tool: str, reply: str) -> str:
    refused = _looks_like_refusal(reply)
    if intended_tool == "refuse":
        return "Pass" if refused else "Fail"
    return "Pass" if not refused else "Fail"


rows = []
for case in EVAL_CASES:
    outcome = run_assistant(case["question"])
    expected = case["expected"]()
    rows.append(
        {
            "Question": case["question"],
            "Intended Tool": case["intended_tool"],
            "LLM Reply": outcome.reply,
            "Math Match (Pass/Fail)": math_match(case["intended_tool"], expected, outcome.reply),
            "Refusal Match (Pass/Fail)": refusal_match(case["intended_tool"], outcome.reply),
            "Latency (ms)": round(outcome.latency_ms, 2),
        }
    )

if pd is not None:
    results_df = pd.DataFrame(rows)
    try:
        from IPython.display import display

        with pd.option_context("display.max_colwidth", 120, "display.max_rows", 20):
            display(results_df)
    except Exception:
        print(results_df.to_string(index=False))
else:
    headers = list(rows[0].keys())
    print("| " + " | ".join(headers) + " |")
    print("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        print("| " + " | ".join(str(row[h]).replace("|", "/") for h in headers) + " |")

math_passes = sum(1 for row in rows if row["Math Match (Pass/Fail)"] == "Pass")
refusal_passes = sum(1 for row in rows if row["Refusal Match (Pass/Fail)"] == "Pass")
print(f"\nMath Match: {math_passes}/{len(rows)}")
print(f"Refusal Match: {refusal_passes}/{len(rows)}")
print(f"Dispatcher: {'mock' if USE_MOCK else 'openai'}")

,Question,Intended Tool,LLM Reply,Math Match (Pass/Fail),Refusal Match (Pass/Fail),Latency (ms)
0,How much DK yarn do I need for a 50 x 60cm blanket in stockinette?,calculate_yarn_quantity,You will need the following for your 50 x 60 cm blanket in stockinette using DK yarn:\n\n- **total_meters**: 1200.0\...,Pass,Pass,3615.98
1,I'm making a 100 x 150cm afghan in bulky yarn using a cable stitch. How much yarn?,calculate_yarn_quantity,"You will need a total of **5250.0 meters** of yarn for your afghan, which means you should estimate about **54 balls...",Pass,Pass,2778.56
2,"Need yarn estimates for a 20x20cm dishcloth in worsted weight, seed stitch.",calculate_yarn_quantity,"For your 20x20 cm dishcloth in worsted weight using seed stitch, here are the yarn estimates:\n\n- **total_meters**:...",Pass,Pass,3180.43
3,How many balls of fingering yarn for a 40x180cm scarf in garter stitch?,calculate_yarn_quantity,"For your 40x180 cm scarf in garter stitch using fingering yarn, here are the details:\n\n- **total_meters**: 4140.0\...",Pass,Pass,2201.99
4,What needle size should I use for worsted yarn?,recommend_needle_size,"For worsted yarn, you should use a needle size in the range of:\n\n- **Metric**: ""4.5 - 5.5"" mm\n- **US Size**: ""7 -...",Pass,Pass,2008.95
5,I want to knit a drapey lace shawl using fingering yarn. What needles?,recommend_needle_size,"For your drapey lace shawl using fingering yarn, I recommend the following needle sizes:\n\n- **metric_mm**: ""4.25""\...",Pass,Pass,2050.81
6,I'm making stiff amigurumi toys in DK yarn. I need a firm fabric. Needle size?,recommend_needle_size,"For your DK yarn and firm fabric, I recommend using a needle size of 3.25 mm (metric) or below 5 (US size). For a fi...",Pass,Pass,2346.25
7,What needles for super bulky yarn to make a standard balanced scarf?,recommend_needle_size,I'm here to help with your knitting! Here are the recommended needle sizes for super bulky yarn to create a standard...,Pass,Pass,3818.34
8,"My pattern says 22 stitches per 10cm, but my swatch has 24 stitches. What's wrong?",troubleshoot_tension,"It looks like your swatch is showing a gauge that is too tight. Here are the details:\n\n- **Diagnosis**: ""Too tight...",Pass,Pass,2562.09
9,"Target gauge is 18 stitches, but I'm getting 15 stitches per 10cm. Help!",troubleshoot_tension,"It looks like your gauge is off! Here's the diagnosis:\n\n- **diagnosis**: ""Too loose""\n- **deviation_percentage**: ...",Pass,Pass,3110.74



Math Match: 15/15
Refusal Match: 15/15
Dispatcher: openai
